In [1]:
import os
import pickle

import numpy as np
import  pandas as pd

import re
import random
import ast
random.seed(0)
np.random.seed(0)

In [2]:
random.seed(0)
#dfaug.sample()

In [3]:
def col_augmentor(src, dstl):
    '''
    Returns the new col which can be inserted in a col-oriented table
    Input : src - list of things obtained from source table
    Output : dstl - number of col in the destination table
    '''
    insertion = -50
    if len(src) > dstl:
        insertion  = src[:dstl]

    elif len(src) < dstl:
        diff = dstl - len(src) 

        nvs = []
        for s in src:
            try: nvs.append(float(find_num(s)))
            except:pass

        if len(nvs) > 0:
            try:
                mean = np.mean(nvs).round(2)
                std = np.std(nvs).round(2)
            except:
                return -50
            # if std != 0:
            #     insertion = list(src) + list(np.random.normal(mean, std, diff).round(2).astype(str))
            # else:
            insertion = list(src) + list(np.random.normal(mean, std, diff).round(2).astype(str))
    elif len(src) == dstl:
        insertion  = src
        
    return insertion

In [4]:
# row augmentation rules
def row_augmentor(src, dstl):
    '''
    Returns the new row which can be inserted in a row-oriented table
    Input : src - list of things obtained from source table
    Output : dstl - number of rows in the destination table
    '''
    if len(src) < dstl:
        diff =  dstl - len(src)  
        nvs = [find_num(s) for s in src if find_num(s) is not None]
        if len(nvs) > 0:
            try:
                mean = np.mean(nvs).round(2)
                std = np.std(nvs).round(2)
            except:
                return -50
            
            if std != 0:
                insertion = src + list(np.random.normal(mean, std, diff).round(2).astype(str))
            # dstl.append(insertion)
            else:
                insertion = list(src) + list(np.random.normal(mean, std, diff).round(2).astype(str))
        else:
#             print('nhi ho rha')
            return -50
            pass

    elif len(src) > dstl:
        insertion  = src[:dstl]

    elif len(src) == dstl:
        insertion = src
    
    return insertion

In [5]:
def showtable(act_table):
    '''
    Input: table_dict['act_table']
    Return: rendered dataframe of actual table
    '''
    display(pd.DataFrame(act_table))

In [6]:
def get_augmentation(table_dict, prop_id):
        
    '''
    Function to obtain the row to be augmented
    Inputs: 
        table_dict having table inside key act_table
        prop_id: ID of the property being augmented
    
    '''
    
    if table_dict['prop_orient'] == 'row':
        return table_dict['act_table'][table_dict['row_label'].index(prop_id)]
    
    elif table_dict['prop_orient']  == 'col':
        return list(np.array(table_dict['act_table'])[:,table_dict['col_label'].index(prop_id)])
    
    else:
        print("CHECK YOUR TABLE DICTIONARY for existence of key 'prop_orient'")

In [7]:
def find_num(string):
    #remove tabs or unnecessary spaces
    try:
        string = string.strip()
    except:
        pass
    #e.g. already in int or float form: 12.5 -> 12.5
    try:
        if string[0] == '-':
            return float(string[1:])*(-1)
        else:
            return float(string)
    except:
        pass
    # e.g. 99.1x10-7
    range_regex = re.compile('^\d+\.?\d*\s*x\s*\d+\.?\d*[-|+]?\s*\d+\.?\d*$')
    try:
#         print('alla')
        ranges_ut = range_regex.search(string).group().split('x')
        ranges = [x.strip() for x in ranges_ut]
        #print(f'ranges = {ranges}')
        if '-' in ranges[1]:
            numu = ranges[1].split('-')
#             print(ranges)
#             print(float(numu[0]), float(numu[1]))
            num = float(ranges[0]) * pow(float(numu[0]), (-float(numu[1])))
            formatted_result = "{:.3e}".format(num)
            try:
                # Try to convert using literal_eval
                return float(formatted_result)
            except (ValueError, SyntaxError):
                # If literal_eval fails, return None or handle the error as needed
                return None
        elif '+' in ranges[1]:
            numu = ranges[1].split('+')
            num = float(ranges[0]) * pow(float(numu[0]), (float(numu[1])))
#             print(num)
            return num
        elif ranges[1].startswith('10') and 3<=len(ranges[1])<=4 and ranges[1].isdigit():
            numu0 = 10
            numu1 = ranges[1][2:]
            num = float(ranges[0]) * pow(float(numu0), (float(numu1)))
            return num
        num = float(ranges[0]) * float(ranges[1])
        formatted_result = "{:.3e}".format(num)
        return formatted_result
    except:
        pass
    #e.g. 12.5 - 13.5 -> 12.5
    range_regex = re.compile('^\d+\.?\d*\s*-\s*\d+\.?\d*')
    try:
        ranges = range_regex.search(string).group().split('-')
        num = float(ranges[0])
#         print(num)
        return num
    except:
        pass
#     print('alla')
    #e.g. 12.2 (5.2) -> 12.2
    bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
    try:
        extracted_value = float(bracket_regex.search(string).group(1))
        return float(extracted_value)
    except:
        pass
    #e.g. 12.3 ± 0.5 -> 12.3
    plusmin_regex = re.compile('^(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')
    try:
        extracted_value = float(plusmin_regex.search(string).group(1))
        return extracted_value
    except AttributeError:
        pass
    #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
    lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
    try:
        extracted_value = lessthan_roughly_regex.search(string).group()
        num_regex = re.compile('\d+\.*\d*')
        extracted_value = num_regex.search(extracted_value).group()
        return float(extracted_value)
    except:
        pass
    # e.g. 0.4:0.6 (ratios)
    if ':' in string:
        split = string.split(":")
        try:
            extracted_value = round(float(split[0])/float(split[1]), 3)
            return extracted_value
        except:
            pass
    # e.g. 220 [29] --> 220.0, where citations given, rejecting ab 220 [29] or 7 220 [29]
    if '[' in string and ']' in string:
        try:
            extracted_value = string[:string.index('[')]
            return float(extracted_value)
        except:
            pass
    # e.g. 723 (first peak or other text) --> 723.0
    if '(' in string and ')' in string:
        try:
            extracted_value = string[:string.index('(')]
            return float(extracted_value)
        except:
            pass
    # e.g. '150K' or '150 degC' --> 150.0 but not '1350degC/2h' or 'njn 150 njn'
    try:
        exact_number_regex = re.compile('^(\d+\.?\d*)\s*[a-zA-Z]+$')
        # Using search to find the first occurrence of the pattern in the string
        match = exact_number_regex.search(string)
#         print('Match')
#         print(match)
        
        # If a match is found, extract the numeric part and convert to float
        if match:
            numeric_part = match.group(1)
            extracted_value = float(numeric_part)
            return extracted_value
    except:
        pass
    return None

In [8]:
# aug_dict = pickle.load(open('aug_dict.pkl', 'rb'))
data = pickle.load(open('../data/new_train_prop_data_abbe_changed.pkl', 'rb'))

In [9]:
print(data[0].keys())
print(len(data))

dict_keys(['act_table', 'caption', 'row_label', 'col_label', 'edge_list', 'pii', 't_idx', 'regex_table', 'num_rows', 'num_cols', 'num_cells', 'comp_table', 'input_ids', 'attention_mask', 'caption_input_ids', 'caption_attention_mask', 'footer', 'gid_row_label', 'gid_col_label', 'sum_less_100', 'prop_table', 'anno_type', 'augment'])
2009


In [10]:
import  ast

In [11]:
# dfaug = pd.read_csv('/home/scai/msr/aiy217586/all_prop_semi_final/train_model/code/prop_augment_info.csv')
dfaug = pd.read_csv('../data/prop_augment_info.csv')
dfaug['freq'][18] = 9
# dfaug['friend_id'] =  dfaug['friend_id'].apply(lambda x: ast.literal_eval(x))
# dfaug.friend_id.iloc[0]

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [12]:
# dfaug = pd.read_csv('/home/scai/msr/aiy217586/all_prop_semi_final/train_model/code/prop_augment_info.csv')
dfaug = pd.read_csv('../data/prop_augment_info.csv')
dfaug['freq'][18] = 9
dfaug.friend_id.apply(lambda x: ast.literal_eval(x))
dfaug['friend_id'][0] = [4, 13, 20, 17, 18]
dfaug['friend_id'][1] = [4, 5, 20, 17, 18]
dfaug['friend_id'][2] = [5, 8, 11, 13, 21]
dfaug['friend_id'][3] = [4, 5, 7, 8, 12, 18]
dfaug['friend_id'][4] = [5]
dfaug['friend_id'][5] = [13, 20, 17, 18]
dfaug['friend_id'][6] = [13, 5, 14, 20]
dfaug['friend_id'][7] = [5, 13, 17, 18]
dfaug['friend_id'][8] = [5, 13, 4, 6, 19, 20, 10, 21, 9, 12, 11]
dfaug['friend_id'][9] = [5, 20, 13, 6, 4, 8, 21, 9, 12, 7]
dfaug['friend_id'][10] = [5, 13, 4, 6]
dfaug['friend_id'][11] = [5, 13, 18, 14]
dfaug['friend_id'][12] = [5, 13, 17, 14]
dfaug['friend_id'][13] = [4, 5, 20, 17, 18]
dfaug['friend_id'][14] = [5, 13, 20, 17, 18]
dfaug['friend_id'][15] = [4, 19, 8, 10, 11, 12, 21]
dfaug['friend_id'][16] = [4, 19, 8, 10, 11, 9, 21]
dfaug['friend_id'][17] = [15, 6, 5]
dfaug['friend_id'][18] = [6]

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until
/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """
/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.o

In [13]:
dfaug

,prop_names,freq,prop_id,friend_id,10*x^0.65,new_freq
0,GlassTransitionTg,497,5,"[4, 13, 20, 17, 18]",566.0,566
1,CrystallizationTemp,418,13,"[4, 5, 20, 17, 18]",506.0,506
2,Density,359,4,"[5, 8, 11, 13, 21]",458.0,458
3,RefractiveIndex,174,6,"[4, 5, 7, 8, 12, 18]",286.0,286
4,ActivationEnergy,119,22,[5],224.0,224
5,MeltingTemp,118,14,"[13, 20, 17, 18]",223.0,223
6,ExpansionCoeff,80,19,"[13, 5, 14, 20]",173.0,173
7,LiquidusTemperature,76,20,"[5, 13, 17, 18]",167.0,167
8,YoungsModulus,74,8,"[5, 13, 4, 6, 19, 20, 10, 21, 9, 12, 11]",165.0,165
9,VickersHardness,55,10,"[5, 20, 13, 6, 4, 8, 21, 9, 12, 7]",136.0,136


In [14]:
dfaug[['prop_names','freq', 'new_freq']]

,prop_names,freq,new_freq
0,GlassTransitionTg,497,566
1,CrystallizationTemp,418,506
2,Density,359,458
3,RefractiveIndex,174,286
4,ActivationEnergy,119,224
5,MeltingTemp,118,223
6,ExpansionCoeff,80,173
7,LiquidusTemperature,76,167
8,YoungsModulus,74,165
9,VickersHardness,55,136


In [15]:
dfaug['difference']  = dfaug.new_freq - dfaug.freq

In [16]:
# for prop_id in range(4, 23):
#     inde = dfaug[dfaug['prop_id']==prop_id].index[0]
#     print(inde)
    
for prop_id in range(4, 23):
    if dfaug[dfaug['prop_id'] == prop_id].empty:
        print(f"Missing prop_id: {prop_id}")

Missing prop_id: 16


In [17]:
# dfaug[dfaug['prop_id']==4].index[0]
# dfaug

In [18]:
random.seed(0)
np.random.seed(0)
same_density_source = []
density_source = []
pii_density_source = []
for prop_id in range(4, 23):
    if prop_id == 16: #dielctric const removed
        continue
    #Collecting the sources
    sources = []
    inde = dfaug[dfaug['prop_id']==prop_id].index[0]
    diff = dfaug['difference'][inde]
    for idx, table in enumerate(data):
        if table['prop_table']:
            if prop_id in table['row_label'] + table['col_label']:
                sources.append(get_augmentation(table, prop_id))
                if prop_id == 4:
                    density_source.append(get_augmentation(table, prop_id))
                    pii_density_source.append((data[idx]['pii'], data[idx]['t_idx']))
    
    if prop_id == 4:
        same_density_source = sources
                
    #Collecting the destination
    destinations_row = []
    destinations_col = []
    f_prop_id = dfaug['friend_id'][inde]
#     print(f'prop_id == {prop_id}')
#     print(f'f_prop_id  == {f_prop_id}')
#     print(f'type of f_prop_id == {type(f_prop_id)}')
    cou = 0
    for idx, table in enumerate(data):
        if table['prop_table']:
            row_col = table['row_label'] + table['col_label']
            com_ele = [elem for elem in f_prop_id if elem in row_col]
#             if 6 in row_col:
#                 cou += 1
#                 print(row_col)
#             print(f'com_ele  ==  {com_ele}')
#             print()
            if any(com_ele):
                if prop_id not in row_col:
                    if table['prop_orient'] == 'row':
                        destinations_row.append(table)
                    else:
                        destinations_col.append(table)
    
#     print(f'cou == {cou}')
                        
    destinations = destinations_row + destinations_col
#     print(len(destinations))
#     print(diff)
#     print()
    dst_idx = random.sample(range(0, len(destinations)), diff)
#     if prop_id==4:
#         print(dst_idx)
    assert diff == len(dst_idx)
    
    
    sid = 0
#     for source in sources: #this is the error
    while sid<diff:
        dst = destinations[dst_idx[sid]]
        if dst['prop_orient'] == 'row':
            insertion = row_augmentor(sources[sid%len(sources)], len(dst['col_label']))
            if insertion == -50:
#                     print(prop_id)
#                     print(source)
#                     print(len(dst['col_label']))
#                     break
                sid +=1
                continue
            else:
                dst['act_table'].append(insertion)
                dst['augment'][0].append(1)
                dst['row_label'].append(prop_id)
                dst['num_rows'] = dst['num_rows'] + 1
        if dst['prop_orient'] == 'col':
            insertion = col_augmentor(sources[sid%len(sources)], len(dst['row_label']))
            if insertion == -50:
#                     print(prop_id)
#                     print(source)
#                     print(len(dst['col_label']))
#                     break
                sid +=1
                continue
            trans_table = list(map(list, zip(*dst['act_table'])))
            trans_table.append(insertion)
            dst['act_table'] = list(map(list, zip(*trans_table)))
            dst['augment'][1].append(1)
            dst['col_label'].append(prop_id)
            dst['num_cols'] = dst['num_cols'] + 1

        dst['num_cells'] = dst['num_rows'] * dst['num_cols']

        if not (dst['num_rows']==len(dst['act_table']) and dst['num_cols']==len(dst['act_table'][0])):
            print('Big Error')
            print(dst)
            print()

        sid += 1

In [19]:
# print(len(density_source))
# for inde, source in enumerate(density_source):
#     print(source)
#     print(pii_density_source[inde])
#     print()

In [20]:
# for d in data:
#     # Check if the sum of d['augment'][0] and d['augment'][1] is greater than 0
#     if sum(d['augment'][0] + d['augment'][1]) > 0:
#         # Use zip to iterate over d['col_label'] and d['augment'][1] simultaneously
#         for col_label, augment in zip(d['col_label'], d['augment'][1]):
#             # Check if col_label is 4 and augment is 1
#             if col_label == 4 and augment == 1:
#                 print(d['augment'], d['row_label'], d['col_label'])
#                 display(pd.DataFrame(d['act_table']))
#                 # Uncomment the next line to display the DataFrame
#                 # display(pd.DataFrame(d['act_table']))
#                 # Exit the loop after finding the first match


In [21]:
pickle.dump(data, open('../data/new_augmented_train_data_verified.pkl', 'wb'))